In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed"

movies = pd.read_csv(raw_dir / "movies.csv")
ratings = pd.read_csv(raw_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"], unit="s", utc=True
)

train_end = pd.Timestamp("2009-11-17 01:21:52+00:00")
validation_end = pd.Timestamp("2016-03-22 08:26:11+00:00")

train_ratings = ratings.loc[
    ratings["rated_at"] <= train_end
].copy()

validation_ratings = ratings.loc[
    (ratings["rated_at"] > train_end)
    & (ratings["rated_at"] <= validation_end)
].copy()

assert len(train_ratings) == 60502
assert len(validation_ratings) == 20167

In [9]:
training_feedback = train_ratings.copy()
training_feedback["liked"] = (
    training_feedback["rating"] >= 4.0
).astype(int)

user_movie_matrix = (
    training_feedback
    .pivot(index="userId", columns="movieId", values="liked")
    .fillna(0)
    .astype(int)
)

movie_user_matrix = user_movie_matrix.T

movie_popularity = (
    train_ratings
    .groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        validate="one_to_one"
    )
    .sort_values(
        ["training_rating_count", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

In [10]:
training_plot_catalog = pd.read_csv(
    processed_dir / "training_plot_catalog.csv"
)

assert training_plot_catalog["movieId"].is_unique
assert set(training_plot_catalog["movieId"]) == set(
    train_ratings["movieId"]
)

training_plot_catalog["has_plot"] = (
    training_plot_catalog["plot"]
    .fillna("")
    .str.strip()
    .ne("")
)

text_movies = (
    training_plot_catalog.loc[training_plot_catalog["has_plot"]]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2"
)

plot_tfidf = tfidf_vectorizer.fit_transform(text_movies["plot"])

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))
print("CF matrix shape:", movie_user_matrix.shape)
print("Movies with plots:", len(text_movies))
print("TF-IDF matrix shape:", plot_tfidf.shape)

Training ratings: 60502
Validation ratings: 20167
CF matrix shape: (6233, 381)
Movies with plots: 5036
TF-IDF matrix shape: (5036, 31609)


In [11]:
def prepare_hybrid_scores(
    user_id, training_data, cf_matrix, popularity, text_movies, text_matrix
):
    history = training_data.loc[training_data["userId"] == user_id]
    liked_ids = history.loc[history["rating"] >= 4.0, "movieId"]

    candidates = popularity.loc[
        ~popularity["movieId"].isin(history["movieId"])
    ].copy()

    candidates["cf_score"] = 0.0
    candidates["text_score"] = 0.0

    if liked_ids.empty:
        return candidates

    cf_values = cosine_similarity(
        cf_matrix.loc[liked_ids], cf_matrix
    ).mean(axis=0)

    candidates["cf_score"] = (
        candidates["movieId"]
        .map(pd.Series(cf_values, index=cf_matrix.index))
        .fillna(0.0)
    )

    text_rows = np.flatnonzero(
        text_movies["movieId"].isin(liked_ids).to_numpy()
    )

    if len(text_rows) > 0:
        text_values = cosine_similarity(
            text_matrix[text_rows], text_matrix
        ).mean(axis=0)

        candidates["text_score"] = (
            candidates["movieId"]
            .map(pd.Series(text_values, index=text_movies["movieId"]))
            .fillna(0.0)
        )

    return candidates


def rank_hybrid(candidate_scores, plot_weight=0.10, n=10):
    if not 0 <= plot_weight <= 1:
        raise ValueError("plot_weight must be between 0 and 1.")

    recommendations = candidate_scores.copy()
    text_max = recommendations["text_score"].max()

    # Preserve CF when plots contribute no evidence
    if plot_weight == 0 or text_max <= 0:
        recommendations["score"] = recommendations["cf_score"]
    else:
        cf_max = recommendations["cf_score"].max()

        cf_scaled = (
            recommendations["cf_score"] / cf_max
            if cf_max > 0 else 0.0
        )
        text_scaled = recommendations["text_score"] / text_max

        recommendations["score"] = (
            (1 - plot_weight) * cf_scaled
            + plot_weight * text_scaled
        )

    return (
        recommendations
        .sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True]
        )
        .head(n)
        .reset_index(drop=True)
    )

In [12]:
from tqdm import tqdm

plot_weights = [0.0, 0.10, 0.25, 0.50]
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= 4.0
]
training_users = set(train_ratings["userId"])

groups = validation_positives.groupby("userId")
evaluation_rows = []

for user_id, positives in tqdm(
    groups, total=groups.ngroups, desc="Evaluating hybrid"
):
    candidate_scores = prepare_hybrid_scores(
        user_id,
        train_ratings,
        movie_user_matrix,
        movie_popularity,
        text_movies,
        plot_tfidf
    )

    positive_ids = set(positives["movieId"])

    for weight in plot_weights:
        recommendations = rank_hybrid(
            candidate_scores, plot_weight=weight, n=10
        )

        hits = len(
            positive_ids.intersection(recommendations["movieId"])
        )

        evaluation_rows.append({
            "plot_weight": weight,
            "userId": user_id,
            "positive_movie_count": len(positive_ids),
            "hits_at_10": hits,
            "recall_at_10": hits / len(positive_ids),
            "has_training_history": user_id in training_users
        })

hybrid_results = pd.DataFrame(evaluation_rows)

Evaluating hybrid: 100%|██████████| 165/165 [00:04<00:00, 33.19it/s]


In [13]:
saved_cf = pd.read_csv(
    processed_dir / "item_based_validation_results.csv"
)

baseline_check = hybrid_results.loc[
    hybrid_results["plot_weight"] == 0
].merge(
    saved_cf,
    on="userId",
    how="outer",
    suffixes=("_new", "_saved"),
    indicator=True,
    validate="one_to_one"
)

assert baseline_check["_merge"].eq("both").all()

for column in ["positive_movie_count", "hits_at_10"]:
    assert (
        baseline_check[f"{column}_new"]
        == baseline_check[f"{column}_saved"]
    ).all()

assert np.allclose(
    baseline_check["recall_at_10_new"],
    baseline_check["recall_at_10_saved"]
)

print("Zero-weight CF evaluation matches the saved baseline.")

Zero-weight CF evaluation matches the saved baseline.


In [14]:
summary_rows = []

for weight, results in hybrid_results.groupby("plot_weight"):
    has_history = results["has_training_history"]

    summary_rows.append({
        "plot_weight": weight,
        "users": len(results),
        "mean_recall_at_10": results["recall_at_10"].mean(),
        "recall_with_history": (
            results.loc[has_history, "recall_at_10"].mean()
        ),
        "recall_without_history": (
            results.loc[~has_history, "recall_at_10"].mean()
        ),
        "total_hits_at_10": results["hits_at_10"].sum()
    })

hybrid_comparison = pd.DataFrame(summary_rows)

display(
    hybrid_comparison.style.format({
        "plot_weight": "{:.2f}",
        "mean_recall_at_10": "{:.4%}",
        "recall_with_history": "{:.4%}",
        "recall_without_history": "{:.4%}"
    })
)

hybrid_results.to_csv(
    processed_dir / "hybrid_validation_results.csv", index=False
)
hybrid_comparison.to_csv(
    processed_dir / "hybrid_validation_comparison.csv", index=False
)

,plot_weight,users,mean_recall_at_10,recall_with_history,recall_without_history,total_hits_at_10
0,0.00,165,4.4762%,6.1086%,4.1847%,326
1,0.10,165,4.4794%,6.1299%,4.1847%,325
2,0.25,165,4.4542%,5.9635%,4.1847%,324
3,0.50,165,4.3560%,5.3155%,4.1847%,319


In [15]:
warm_results = hybrid_results.loc[
    hybrid_results["has_training_history"]
]

columns = [
    "userId",
    "positive_movie_count",
    "hits_at_10",
    "recall_at_10"
]

cf_results = warm_results.loc[
    warm_results["plot_weight"] == 0,
    columns
]

blend_results = warm_results.loc[
    warm_results["plot_weight"] == 0.10,
    columns
]

hybrid_user_comparison = cf_results.merge(
    blend_results,
    on="userId",
    suffixes=("_cf", "_hybrid"),
    validate="one_to_one"
)

assert (
    hybrid_user_comparison["positive_movie_count_cf"]
    == hybrid_user_comparison["positive_movie_count_hybrid"]
).all()

hybrid_user_comparison["recall_change"] = (
    hybrid_user_comparison["recall_at_10_hybrid"]
    - hybrid_user_comparison["recall_at_10_cf"]
)

change = hybrid_user_comparison["recall_change"]

print("Users compared:", len(hybrid_user_comparison))
print("Improved recall:", change.gt(0).sum())
print("Worsened recall:", change.lt(0).sum())
print("Unchanged recall:", change.eq(0).sum())

print(
    "Net hit change:",
    (
        hybrid_user_comparison["hits_at_10_hybrid"]
        - hybrid_user_comparison["hits_at_10_cf"]
    ).sum()
)

display(
    hybrid_user_comparison.loc[
        change.ne(0),
        [
            "userId",
            "positive_movie_count_cf",
            "hits_at_10_cf",
            "hits_at_10_hybrid",
            "recall_at_10_cf",
            "recall_at_10_hybrid",
            "recall_change"
        ]
    ]
    .sort_values("recall_change", ascending=False)
    .reset_index(drop=True)
)

hybrid_user_comparison.to_csv(
    processed_dir / "hybrid_010_vs_cf_users.csv",
    index=False
)

Users compared: 25
Improved recall: 2
Worsened recall: 2
Unchanged recall: 21
Net hit change: -1


,userId,positive_movie_count_cf,hits_at_10_cf,hits_at_10_hybrid,recall_at_10_cf,recall_at_10_hybrid,recall_change
0,477,30,0,1,0.000000,0.033333,0.033333
1,68,119,3,4,0.025210,0.033613,0.008403
2,590,119,4,2,0.033613,0.016807,-0.016807
3,543,51,1,0,0.019608,0.000000,-0.019608


In [16]:
def inspect_hybrid_changes(user_id):
    weight = 0.10

    candidates = prepare_hybrid_scores(
        user_id, train_ratings, movie_user_matrix,
        movie_popularity, text_movies, plot_tfidf
    )

    cf_ranked = rank_hybrid(
        candidates, plot_weight=0, n=len(candidates)
    )
    hybrid_ranked = rank_hybrid(
        candidates, plot_weight=weight, n=len(candidates)
    )

    cf_top = set(cf_ranked.head(10)["movieId"])
    hybrid_top = set(hybrid_ranked.head(10)["movieId"])
    changed_ids = cf_top.symmetric_difference(hybrid_top)

    # Use maxima from ALL eligible candidates
    cf_max = candidates["cf_score"].max()
    text_max = candidates["text_score"].max()

    if text_max <= 0:
        raise ValueError("This user has no positive plot scores.")

    changes = candidates.loc[
        candidates["movieId"].isin(changed_ids)
    ].copy()

    cf_ranks = pd.Series(
        np.arange(1, len(cf_ranked) + 1),
        index=cf_ranked["movieId"]
    )
    hybrid_ranks = pd.Series(
        np.arange(1, len(hybrid_ranked) + 1),
        index=hybrid_ranked["movieId"]
    )

    changes["cf_rank"] = changes["movieId"].map(cf_ranks)
    changes["hybrid_rank"] = changes["movieId"].map(hybrid_ranks)

    changes["change"] = np.where(
        changes["movieId"].isin(hybrid_top), "Entered", "Left"
    )

    changes["cf_part"] = (
        (1 - weight) * changes["cf_score"] / cf_max
        if cf_max > 0 else 0.0
    )
    changes["plot_part"] = (
        weight * changes["text_score"] / text_max
    )

    changes["has_plot"] = changes["movieId"].isin(
        text_movies["movieId"]
    )

    positive_ids = set(
        validation_ratings.loc[
            (validation_ratings["userId"] == user_id)
            & (validation_ratings["rating"] >= 4.0),
            "movieId"
        ]
    )
    changes["positive_in_validation"] = (
        changes["movieId"].isin(positive_ids)
    )

    return changes[
        [
            "change", "movieId", "title", "has_plot",
            "cf_rank", "hybrid_rank", "cf_part", "plot_part",
            "positive_in_validation"
        ]
    ].sort_values(["change", "hybrid_rank"])


for user_id in [477, 590]:
    print(f"\nUser {user_id}")

    changes = inspect_hybrid_changes(user_id)

    display(
        changes.style.format({
            "cf_part": "{:.4f}",
            "plot_part": "{:.4f}"
        })
    )

    changes.to_csv(
        processed_dir / f"hybrid_changes_user_{user_id}.csv",
        index=False
    )



User 477


,change,movieId,title,has_plot,cf_rank,hybrid_rank,cf_part,plot_part,positive_in_validation
134,Entered,2329,American History X (1998),True,12,8,0.7930,0.0538,False
26,Entered,608,Fargo (1996),True,11,9,0.7990,0.0421,True
107,Entered,1213,Goodfellas (1990),True,14,10,0.7805,0.0584,False
276,Left,7438,Kill Bill: Vol. 2 (2004),False,7,11,0.8362,0.0000,False
127,Left,912,Casablanca (1942),True,10,12,0.8016,0.0290,False
88,Left,1221,"Godfather: Part II, The (1974)",False,9,16,0.8050,0.0000,False



User 590


,change,movieId,title,has_plot,cf_rank,hybrid_rank,cf_part,plot_part,positive_in_validation
162,Entered,2502,Office Space (1999),True,14,9,0.8353,0.0548,False
139,Entered,1682,"Truman Show, The (1998)",True,11,10,0.8532,0.0351,False
233,Left,2194,"Untouchables, The (1987)",True,10,11,0.8564,0.0297,True
24,Left,50,"Usual Suspects, The (1995)",False,5,14,0.8799,0.0000,True


In [ ]:
cmu_dir = project_root / "data" / "raw" / "MovieSummaries"

cmu_titles = pd.read_csv(
    cmu_dir / "movie.metadata.tsv",
    sep="\t",
    header=None,
    usecols=[0, 2, 3]
).rename(columns={
    0: "wikipedia_movie_id",
    2: "cmu_title",
    3: "release_date"
})

plot_ids = pd.read_csv(
    cmu_dir / "plot_summaries.txt",
    sep="\t",
    header=None,
    usecols=[0]
)[0]

cmu_titles["has_cmu_plot"] = (
    cmu_titles["wikipedia_movie_id"].isin(plot_ids)
)

target_rows = cmu_titles.loc[
    cmu_titles["cmu_title"].str.contains(
        r"usual suspects|godfather|kill bill",
        case=False,
        na=False,
        regex=True
    )
]

display(
    target_rows.sort_values(["cmu_title", "release_date"])
)